<a href="https://colab.research.google.com/github/Burzo572/dsb_project/blob/main/Model_Janko_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project — Flood Flag Prediction in Bangladesh

**Final Modeling Notebook | SW90 (Bahadurabad), SW17 (Chilmari), and 30-station experiment**

This notebook builds on the exploratory data analysis (EDA). The goal is to investigate whether recent rainfall, soil moisture, and river discharge can help identify **positive GFMS flood-related flags**, and to assess how reliably models recognize new transitions into a positive flag.

The experiments use chronological data from **2021–2024**. They begin with simple baselines, progress to Logistic Regression using SW90 and SW17, extend the analysis to 24h/48h/72h GFMS labels, and finally compare models trained on SW90 versus all 30 stations. Thresholds, onset detection, uncertainty, and risk categories are examined because positive labels are relatively rare.

> **Interpretation note:** The dataset's GFMS flags and their 24h/48h/72h names do not by themselves establish that a physical flood was confirmed by a river gauge or that the models provide a verified 24-, 48-, or 72-hour operational warning. Input/label timestamps and source construction must be checked before making that claim.

---

## 1. Setup and data preparation
### Import the required libraries

Load Pandas and NumPy for data preparation, scikit-learn models and scaling, and evaluation metrics. We will use **Precision, Recall, F1-score, confusion matrices, and Average Precision (AP)** to focus on the rare positive class, rather than relying only on accuracy.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score, precision_score, recall_score, f1_score
from sklearn.ensemble import HistGradientBoostingClassifier

### Load the refined SW90 and SW17 dataset

Read the CSV prepared from the EDA and sort observations by station and date. It contains both stations for **2021–2025**; we will restrict modeling to earlier years so that comparisons are consistent and the problematic 2025 period is not included.

In [2]:
# Load data directly from GitHub
url = "https://raw.githubusercontent.com/Burzo572/dsb_project/main/data/Data_SW90_SW17_Original_2021_2025.csv"

df_refined = pd.read_csv(url, parse_dates=["date"])

# Sort chronologically
df_refined = df_refined.sort_values(["station_id", "date"]).reset_index(drop=True)

print("Shape:", df_refined.shape)
display(df_refined.head())

Shape: (3652, 50)


,date,station_id,station_name,rainfall_local_mm,rainfall_upstream_mm,soil_moisture_local,river_discharge_m3s,flood_byStor,rainfall_local_mm_missing,rainfall_upstream_mm_missing,...,flood_within_24h_label_regime,flood_within_48h,flood_within_48h_label_regime,flood_within_72h,flood_within_72h_label_regime,upstream_chain_discharge_lag1d,upstream_reference_discharge_lag2d,upstream_chain_discharge_lag2d,upstream_reference_discharge_lag3d,year
0,2021-01-01,SW17,Chilmari,0.0,0.000000,0.356,3705.81,NaN,False,False,...,observed,0.0,observed,0.0,observed,NaN,NaN,NaN,NaN,2021
1,2021-01-02,SW17,Chilmari,0.0,0.000000,0.348,3646.54,NaN,False,False,...,observed,0.0,observed,0.0,observed,NaN,NaN,NaN,NaN,2021
2,2021-01-03,SW17,Chilmari,0.0,0.000000,0.342,3588.22,NaN,False,False,...,observed,0.0,observed,0.0,observed,NaN,NaN,NaN,NaN,2021
3,2021-01-04,SW17,Chilmari,0.0,0.000000,0.337,3522.71,NaN,False,False,...,observed,0.0,observed,0.0,observed,NaN,NaN,NaN,NaN,2021
4,2021-01-05,SW17,Chilmari,0.0,0.533333,0.336,3466.37,NaN,False,False,...,observed,0.0,observed,0.0,observed,NaN,NaN,NaN,NaN,2021


### Filter 2021–2024 and review missingness

Keep the four modeling years and calculate the percentage of missing observations in candidate predictors for each station. This check explains why the later models primarily use **local rainfall, local soil moisture, and river discharge**: those columns are complete in this subset, unlike some trend or upstream-chain features.

In [3]:
# Select years 2021–2024 for modeling
df_refined_model = df_refined[df_refined["year"].between(2021, 2024)].copy()

# Select potential predictors
features_refined = [
    "rainfall_local_mm",
    "rainfall_upstream_mm",
    "soil_moisture_local",
    "river_discharge_m3s",
    "rainfall_local_mm_trend_ratio",
    "rainfall_upstream_mm_trend_ratio",
    "upstream_chain_discharge_lag1d"]

# Check missing values for each station
missing_refined = (
    df_refined_model
    .groupby("station_id")[features_refined]
    .agg(lambda x: round(x.isna().mean() * 100, 2))
    .T)

print("Dataset shape:", df_refined_model.shape)

display(missing_refined)

Dataset shape: (2922, 50)


station_id,SW17,SW90
rainfall_local_mm,0.00,0.00
rainfall_upstream_mm,0.00,0.00
soil_moisture_local,0.00,0.00
river_discharge_m3s,0.00,0.00
rainfall_local_mm_trend_ratio,24.44,24.44
rainfall_upstream_mm_trend_ratio,0.34,0.34
upstream_chain_discharge_lag1d,100.00,0.00


## 2. Baselines and GFMS flag transitions
### Define onset and continuation for SW90

Use the previous day's 24h GFMS label to describe changes between consecutive labels. An **onset** is a transition from 0 to 1; a **continuation** is 1 followed by 1. Here these terms describe *label transitions*, not independently verified physical flood events. Distinguishing them is essential because a persistence rule cannot identify a new 0→1 transition.

In [4]:
# Select SW90 from the refined dataset
sw90_refined = df_refined_model[df_refined_model["station_id"] == "SW90"].copy()

# Sort by date
sw90_refined = sw90_refined.sort_values("date").reset_index(drop=True)

# Target: next GFMS flood flag
sw90_refined["next_flood_flag"] = (sw90_refined["flood_within_24h"])

# Previous day's target label
sw90_refined["previous_flood_flag"] = (sw90_refined["next_flood_flag"].shift(1))

# Exclude the first row and missing labels
sw90_refined = sw90_refined.dropna(subset=["next_flood_flag", "previous_flood_flag"]).copy()

# Identify flood onsets and continuations
sw90_refined["flood_onset"] = ((sw90_refined["previous_flood_flag"] == 0) & (sw90_refined["next_flood_flag"] == 1))

sw90_refined["flood_continuation"] = ((sw90_refined["previous_flood_flag"] == 1) & (sw90_refined["next_flood_flag"] == 1))

# Results
print("Total rows:", len(sw90_refined))
print("Flood onsets:", sw90_refined["flood_onset"].sum())
print("Flood continuations:", sw90_refined["flood_continuation"].sum())

print("\nFlood labels by year:")
display(sw90_refined.groupby(sw90_refined["date"].dt.year)[["next_flood_flag", "flood_onset", "flood_continuation"]].sum())

print("\nPrevious vs Next GFMS Flag:")
display(pd.crosstab(sw90_refined["previous_flood_flag"],sw90_refined["next_flood_flag"]))

Total rows: 1460
Flood onsets: 35
Flood continuations: 15

Flood labels by year:


,next_flood_flag,flood_onset,flood_continuation
date,,,
2021,13.0,10,3
2022,19.0,12,7
2023,11.0,8,3
2024,7.0,5,2



Previous vs Next GFMS Flag:


next_flood_flag,0.0,1.0
previous_flood_flag,,
0.0,1375,35
1.0,35,15


### Persistence baseline — initial chronological split

Construct a simple benchmark: predict the current positive label whenever the previous label was positive. Evaluate on the later 30% of the time series, without random shuffling. This baseline is useful for measuring how much a model improves beyond the tendency of positive labels to persist across days.

In [5]:
# Keep rows corresponding to 3-day windows
persistence_data = sw90_refined.iloc[1:].copy()

# Chronological 70/30 split
split_idx = int(len(persistence_data) * 0.7)

persistence_test = persistence_data.iloc[split_idx:].copy()

# Actual GFMS labels
y_test_persistence = persistence_test["next_flood_flag"]

# Persistence prediction: previous flag = next prediction
y_pred_persistence = persistence_test["previous_flood_flag"]

print("Test rows:", len(persistence_test))
print("Test period:",persistence_test["date"].min(),"to",persistence_test["date"].max())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_persistence,y_pred_persistence))

print("\nClassification Report:")
print(classification_report(y_test_persistence,y_pred_persistence,zero_division=0))

# How many flood onsets did persistence detect?
onset_test = persistence_test[persistence_test["flood_onset"]]

print("\nFlood onsets in test:", len(onset_test))

print("Onsets detected:",(onset_test["previous_flood_flag"] == 1).sum())

Test rows: 438
Test period: 2023-10-21 00:00:00 to 2024-12-31 00:00:00

Confusion Matrix:
[[426   5]
 [  5   2]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       431
         1.0       0.29      0.29      0.29         7

    accuracy                           0.98       438
   macro avg       0.64      0.64      0.64       438
weighted avg       0.98      0.98      0.98       438


Flood onsets in test: 5
Onsets detected: 0


### Align the persistence baseline with the 3-day model windows

Match the same test dates and 438 observations that will be used by the 3-day Logistic Regression experiment. This alignment ensures that baseline and model metrics refer to **identical target rows**. The bookkeeping variable `target_date` is the next calendar date, but the GFMS label itself is taken from the final input day's record.

In [6]:
# Align persistence with our original 3-day sliding windows

persistence_aligned = sw90_refined[(sw90_refined["date"] >= "2021-01-03") &(sw90_refined["date"] < "2024-12-31")].copy()

# Match the target_date convention from the original modeling
persistence_aligned["target_date"] = (persistence_aligned["date"] + pd.Timedelta(days=1))

persistence_aligned = persistence_aligned.sort_values("target_date").reset_index(drop=True)

# Same chronological 70/30 split
split_idx = int(len(persistence_aligned) * 0.7)

persistence_test_aligned = persistence_aligned.iloc[split_idx:].copy()

y_test_persistence_aligned = (persistence_test_aligned["next_flood_flag"])

y_pred_persistence_aligned = (persistence_test_aligned["previous_flood_flag"])

print("Test rows:", len(persistence_test_aligned))

print("First target date:",persistence_test_aligned["target_date"].min())

print("Last target date:",persistence_test_aligned["target_date"].max())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_persistence_aligned,y_pred_persistence_aligned))

print("\nClassification Report:")
print(classification_report(y_test_persistence_aligned,y_pred_persistence_aligned,zero_division=0))

print("\nOnsets in test:",persistence_test_aligned["flood_onset"].sum())

print("Continuations in test:",persistence_test_aligned["flood_continuation"].sum())

Test rows: 438
First target date: 2023-10-21 00:00:00
Last target date: 2024-12-31 00:00:00

Confusion Matrix:
[[426   5]
 [  5   2]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       431
         1.0       0.29      0.29      0.29         7

    accuracy                           0.98       438
   macro avg       0.64      0.64      0.64       438
weighted avg       0.98      0.98      0.98       438


Onsets in test: 5
Continuations in test: 2


### Seasonal baseline — June–August vs June–September

Test rules that predict a positive flag throughout the selected monsoon months. Including September raises the chance of detecting positives, but also increases false alarms. These seasonal rules provide a transparent comparison for more complex models.

In [7]:
# Seasonal Baseline comparison
for months in [[6, 7, 8], [6, 7, 8, 9]]:
    y_pred = persistence_test_aligned["target_date"].dt.month.isin(months).astype(int)

    print("\nMonths:", months)
    print(confusion_matrix(y_test_persistence_aligned, y_pred))
    print(classification_report(y_test_persistence_aligned, y_pred, zero_division=0))


Months: [6, 7, 8]
[[344  87]
 [  2   5]]
              precision    recall  f1-score   support

         0.0       0.99      0.80      0.89       431
         1.0       0.05      0.71      0.10         7

    accuracy                           0.80       438
   macro avg       0.52      0.76      0.49       438
weighted avg       0.98      0.80      0.87       438


Months: [6, 7, 8, 9]
[[315 116]
 [  1   6]]
              precision    recall  f1-score   support

         0.0       1.00      0.73      0.84       431
         1.0       0.05      0.86      0.09         7

    accuracy                           0.73       438
   macro avg       0.52      0.79      0.47       438
weighted avg       0.98      0.73      0.83       438



### Evaluate seasonal baseline onset detection

Count how many **new positive-flag transitions** each seasonal rule captures. This complements the confusion matrix: a rule can have high Recall simply by predicting positives on many days, so its false-alarm burden must also be considered.

In [8]:
# Flood onsets detected by seasonal baselines

onsets = persistence_test_aligned["flood_onset"]

for months in [[6, 7, 8], [6, 7, 8, 9]]:
    pred = persistence_test_aligned["target_date"].dt.month.isin(months)

    print("Months:", months)
    print("Onsets detected:", (pred & onsets).sum(), "/", onsets.sum())

Months: [6, 7, 8]
Onsets detected: 3 / 5
Months: [6, 7, 8, 9]
Onsets detected: 4 / 5


## 3. Logistic Regression with SW90 + SW17
### Build three-day sliding windows

Create an input row from **three consecutive days** for each station, using local rainfall, soil moisture, and river discharge. Two stations × three variables × three days produce **18 predictors**. The target is the SW90 24h GFMS label associated with the last day of the input window; `target_date` is stored one day later for consistent indexing.

In [9]:
# Prepare SW90 and SW17
sw90_new = df_refined_model[df_refined_model["station_id"] == "SW90"].sort_values("date").reset_index(drop=True)
sw17_new = df_refined_model[df_refined_model["station_id"] == "SW17"].sort_values("date").reset_index(drop=True)

features_new = ["rainfall_local_mm", "soil_moisture_local", "river_discharge_m3s"]

# Create 3-day sliding windows
rows_new = []

for i in range(2, len(sw90_new) - 1):
    row = {"target_date": sw90_new.loc[i, "date"] + pd.Timedelta(days=1),
           "flood_flag": sw90_new.loc[i, "flood_within_24h"]}

    for day in range(3):
        for station, data in [("SW90", sw90_new), ("SW17", sw17_new)]:
            for feature in features_new:
                row[f"{feature}_{station}_day{day+1}"] = data.loc[i-2+day, feature]

    rows_new.append(row)

chunks_new = pd.DataFrame(rows_new)

print("Shape:", chunks_new.shape)
print("Flood labels:", chunks_new["flood_flag"].value_counts())

Shape: (1458, 20)
Flood labels: flood_flag
0.0    1408
1.0      50
Name: count, dtype: int64


### Initial 70/30 Logistic Regression evaluation

Split the windows chronologically, standardize predictors using the **training set only**, then fit Logistic Regression. Compare predictions with persistence on the exact same test rows. Report probability-ranking performance with Average Precision and explicitly count detected onsets to avoid mistaking high overall accuracy for useful flood-flag detection.

In [10]:
# Train / Test split
split_new = int(len(chunks_new) * 0.7)
train_new = chunks_new.iloc[:split_new]
test_new = chunks_new.iloc[split_new:]

X_train_new = train_new.drop(columns=["target_date", "flood_flag"])
X_test_new = test_new.drop(columns=["target_date", "flood_flag"])
y_train_new = train_new["flood_flag"]
y_test_new = test_new["flood_flag"]

# Check that we use the same test dates as Persistence
assert test_new["target_date"].reset_index(drop=True).equals(persistence_test_aligned["target_date"].reset_index(drop=True))
assert y_test_new.reset_index(drop=True).equals(persistence_test_aligned["next_flood_flag"].reset_index(drop=True))

# Standardization
scaler_new = StandardScaler()
X_train_scaled_new = scaler_new.fit_transform(X_train_new)
X_test_scaled_new = scaler_new.transform(X_test_new)

# Logistic Regression
log_new = LogisticRegression(max_iter=1000)
log_new.fit(X_train_scaled_new, y_train_new)

# Predictions and probabilities
y_pred_new = log_new.predict(X_test_scaled_new)
y_prob_new = log_new.predict_proba(X_test_scaled_new)[:, 1]

# Results
print("Train Accuracy:", log_new.score(X_train_scaled_new, y_train_new))
print("Test Accuracy:", log_new.score(X_test_scaled_new, y_test_new))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_new, y_pred_new))
print("\nClassification Report:\n", classification_report(y_test_new, y_pred_new, zero_division=0))
print("PR-AUC (Average Precision):", average_precision_score(y_test_new, y_prob_new))

# Flood onset detection
onsets_new = persistence_test_aligned["flood_onset"].to_numpy()
print("Onsets detected:", y_pred_new[onsets_new].sum(), "/", onsets_new.sum())

Train Accuracy: 0.9568627450980393
Test Accuracy: 0.9794520547945206

Confusion Matrix:
 [[428   3]
 [  6   1]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       431
         1.0       0.25      0.14      0.18         7

    accuracy                           0.98       438
   macro avg       0.62      0.57      0.59       438
weighted avg       0.97      0.98      0.98       438

PR-AUC (Average Precision): 0.35351418990639993
Onsets detected: 0.0 / 5


### Inspect positive-label counts by year

Summarize positive and negative SW90 labels across years. The class imbalance is clear: the positive class is uncommon and particularly sparse in 2024, so performance metrics can change sharply with just a few correct or missed flags.

In [11]:
# Check flood labels by year
floods_by_year = pd.crosstab(chunks_new["target_date"].dt.year, chunks_new["flood_flag"])

display(floods_by_year)

flood_flag,0.0,1.0
target_date,,
2021,349,13
2022,346,19
2023,354,11
2024,359,7


### Year-by-year walk-forward Logistic Regression

For each test year (2022, 2023, and 2024), train only on **previous years**. This is a more realistic evaluation than a random split because it avoids training on future seasons. Report per-year Precision, Recall, F1-score, Average Precision, and detected onsets to assess stability across changing conditions.

In [12]:
# Prepare data
yearly_data = chunks_new.copy()
yearly_data["label_date"] = yearly_data["target_date"] - pd.Timedelta(days=1)

features = chunks_new.columns.drop(["target_date", "flood_flag"])
onsets = sw90_refined.set_index("date")["flood_onset"]

results_yearly = []

for year in [2022, 2023, 2024]:
    train = yearly_data[yearly_data["label_date"].dt.year < year]
    test = yearly_data[yearly_data["label_date"].dt.year == year]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[features])
    X_test = scaler.transform(test[features])

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, train["flood_flag"])

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    onset = test["label_date"].map(onsets).fillna(False).astype(bool).to_numpy()

    results_yearly.append({
        "Test Year": year,
        "Floods": int(test["flood_flag"].sum()),
        "Onsets": int(onset.sum()),
        "Detected Onsets": int(pred[onset].sum()),
        "Precision": precision_score(test["flood_flag"], pred, zero_division=0),
        "Recall": recall_score(test["flood_flag"], pred, zero_division=0),
        "F1": f1_score(test["flood_flag"], pred, zero_division=0),
        "PR-AUC (AP)": average_precision_score(test["flood_flag"], prob)})

display(pd.DataFrame(results_yearly).round(3))

,Test Year,Floods,Onsets,Detected Onsets,Precision,Recall,F1,PR-AUC (AP)
0,2022,19,12,2,0.286,0.211,0.242,0.308
1,2023,11,8,2,0.500,0.455,0.476,0.301
2,2024,7,5,0,0.250,0.143,0.182,0.354


### Year-by-year persistence comparison

Apply the same prior-flag rule separately to each test year. This gives an interpretable F1-score benchmark against which the walk-forward Logistic Regression results can be judged.

In [13]:
# Persistence Baseline - Year by Year

previous_flags = sw90_refined.set_index("date")["previous_flood_flag"]

results_persistence = []

for year in [2022, 2023, 2024]:
    test = yearly_data[yearly_data["label_date"].dt.year == year]

    y_true = test["flood_flag"]
    y_pred = test["label_date"].map(previous_flags)

    results_persistence.append({
        "Year": year,
        "Floods": int(y_true.sum()),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)})

display(pd.DataFrame(results_persistence).round(3))

,Year,Floods,Precision,Recall,F1
0,2022,19,0.368,0.368,0.368
1,2023,11,0.273,0.273,0.273
2,2024,7,0.286,0.286,0.286


### Probabilistic baseline — previous flag and month

Estimate the frequency of positive flags for each `(month, previous flag)` combination using training years only. Apply the learned frequencies to the next year and compute Average Precision. Unlike the binary persistence rule, this baseline gives a **probability-like score**, making its ranking directly comparable with Logistic Regression.

In [14]:
# Probabilistic Baseline - Year by Year

prob_data = yearly_data.copy()
prob_data["month"] = prob_data["label_date"].dt.month
prob_data["previous_flag"] = prob_data["label_date"].map(previous_flags)

results_prob = []

for year in [2022, 2023, 2024]:
    train = prob_data[prob_data["label_date"].dt.year < year]
    test = prob_data[prob_data["label_date"].dt.year == year].copy()

    # Learn probabilities from previous years
    probabilities = train.groupby(["month", "previous_flag"])["flood_flag"].mean()

    # Apply probabilities to the test year
    test = test.join(probabilities.rename("prob"), on=["month", "previous_flag"])
    test["prob"] = test["prob"].fillna(test["previous_flag"].map(train.groupby("previous_flag")["flood_flag"].mean()))
    test["prob"] = test["prob"].fillna(train["flood_flag"].mean())

    ap = average_precision_score(test["flood_flag"], test["prob"])
    results_prob.append({"Year": year, "PR-AUC (AP)": round(ap, 3)})

display(pd.DataFrame(results_prob))

,Year,PR-AUC (AP)
0,2022,0.252
1,2023,0.253
2,2024,0.168


## 4. Comparing 24h, 48h, and 72h GFMS labels
### Inspect target availability and positive-label counts

Count positive labels for all three named horizons and check for missing targets. Longer-horizon labels contain more positive rows, which may change the difficulty and apparent performance of classification. The horizon labels remain **GFMS-defined categories**, not yet verified advance-warning lead times.

In [15]:
# SW90 - Different flood prediction horizons
sw90_horizons = df_refined_model[df_refined_model["station_id"] == "SW90"].copy()

labels = ["flood_within_24h", "flood_within_48h", "flood_within_72h"]

# Number of positive GFMS labels by year
display(sw90_horizons.groupby("year")[labels].sum())

# Check missing labels
print("Missing labels:")
print(sw90_horizons[labels].isna().sum())

,flood_within_24h,flood_within_48h,flood_within_72h
year,,,
2021,13.0,23.0,32.0
2022,19.0,31.0,42.0
2023,11.0,19.0,26.0
2024,7.0,12.0,17.0


Missing labels:
flood_within_24h    0
flood_within_48h    0
flood_within_72h    0
dtype: int64


### Attach 48h and 72h labels to the existing windows

Reuse the same 18 predictors and three-day windows, changing only the target labels. Mapping by `label_date` prevents accidental row-order mismatches between sliding windows and the original SW90 records.

In [16]:
# Add 48h and 72h labels to our existing sliding windows
chunks_extended = chunks_new.copy()
chunks_extended["label_date"] = chunks_extended["target_date"] - pd.Timedelta(days=1)

sw90_labels = sw90_horizons.set_index("date")

for h in [48, 72]:
    chunks_extended[f"flood_{h}h"] = chunks_extended["label_date"].map(sw90_labels[f"flood_within_{h}h"])

# Check the new labels
print("Shape:", chunks_extended.shape)
print("\nPositive labels:")
print(chunks_extended[["flood_flag", "flood_48h", "flood_72h"]].sum())

print("\nMissing labels:")
print(chunks_extended[["flood_flag", "flood_48h", "flood_72h"]].isna().sum())

Shape: (1458, 23)

Positive labels:
flood_flag     50.0
flood_48h      85.0
flood_72h     117.0
dtype: float64

Missing labels:
flood_flag    0
flood_48h     0
flood_72h     0
dtype: int64


### Walk-forward models for the 48h and 72h targets

Train Logistic Regression on earlier years and evaluate each later test year. A short cutoff near the year boundary reduces the risk that longer-horizon labels overlap the evaluation period. Compare detected positives, false alarms, F1-score, and Average Precision across years and horizons.

In [17]:
# Logistic Regression - 48h and 72h, year by year

results_extended = []
feature_cols = chunks_new.columns.drop(["target_date", "flood_flag"])

for h in [48, 72]:
    for year in [2022, 2023, 2024]:

        train = chunks_extended[chunks_extended["label_date"].dt.year < year].copy()
        test = chunks_extended[chunks_extended["label_date"].dt.year == year].copy()

        # Avoid overlap between training labels and test period
        train = train[train["label_date"] + pd.Timedelta(hours=h) < test["label_date"].min()]

        X_train = train[feature_cols]
        X_test = test[feature_cols]
        y_train = train[f"flood_{h}h"]
        y_test = test[f"flood_{h}h"]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_scaled, y_train)

        pred = model.predict(X_test_scaled)
        prob = model.predict_proba(X_test_scaled)[:, 1]

        results_extended.append({
            "Horizon": f"{h}h",
            "Year": year,
            "Floods": int(y_test.sum()),
            "Detected": int(((pred == 1) & (y_test == 1)).sum()),
            "False Alarms": int(((pred == 1) & (y_test == 0)).sum()),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "F1": f1_score(y_test, pred, zero_division=0),
            "PR-AUC (AP)": average_precision_score(y_test, prob)})

comparison_extended = pd.DataFrame(results_extended)
display(comparison_extended.round(3))

,Horizon,Year,Floods,Detected,False Alarms,Precision,Recall,F1,PR-AUC (AP)
0,48h,2022,31,10,8,0.556,0.323,0.408,0.403
1,48h,2023,19,7,9,0.438,0.368,0.400,0.355
2,48h,2024,12,4,9,0.308,0.333,0.320,0.314
3,72h,2022,42,11,5,0.688,0.262,0.379,0.445
4,72h,2023,26,8,9,0.471,0.308,0.372,0.413
5,72h,2024,17,8,6,0.571,0.471,0.516,0.377


### Count onsets and continuations at longer horizons

For 48h and 72h GFMS labels, classify positive rows as either new 0→1 transitions or continuations from an already-positive previous day. This matters because expanding the horizon can increase the number of **continuation** labels without increasing the number of new transitions.

In [18]:
# Onsets and continuations - 48h and 72h

for h in [48, 72]:
    col = f"flood_within_{h}h"
    data = sw90_horizons.sort_values("date").copy()

    data["previous_flag"] = data[col].shift(1)
    data["onset"] = (data["previous_flag"] == 0) & (data[col] == 1)
    data["continuation"] = (data["previous_flag"] == 1) & (data[col] == 1)

    print(f"\nGFMS {h}h:")
    display(data.groupby("year")[[col, "onset", "continuation"]].sum().loc[2022:2024])


GFMS 48h:


,flood_within_48h,onset,continuation
year,,,
2022,31.0,11,20
2023,19.0,7,12
2024,12.0,5,7



GFMS 72h:


,flood_within_72h,onset,continuation
year,,,
2022,42.0,11,31
2023,26.0,7,19
2024,17.0,5,12


### Evaluate onset recall for 48h and 72h

Repeat the walk-forward Logistic Regression fitting and explicitly measure how many label onsets were captured. The results help determine whether positive predictions correspond to new transitions or mainly to consecutive days within an already-positive period.

In [19]:
# Flood Onset Detection - Logistic Regression 48h and 72h

onset_results = []
features = chunks_new.columns.drop(["target_date", "flood_flag"])

for h in [48, 72]:
    col = f"flood_within_{h}h"
    station = sw90_horizons.sort_values("date").copy()
    station["onset"] = (station[col].shift(1) == 0) & (station[col] == 1)
    onset_dates = station.set_index("date")["onset"]

    for year in [2022, 2023, 2024]:
        train = chunks_extended[chunks_extended["label_date"].dt.year < year].copy()
        test = chunks_extended[chunks_extended["label_date"].dt.year == year].copy()
        train = train[train["label_date"] + pd.Timedelta(hours=h) < test["label_date"].min()]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(train[features])
        X_test = scaler.transform(test[features])

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, train[f"flood_{h}h"])
        pred = model.predict(X_test)

        onsets = test["label_date"].map(onset_dates).fillna(False).to_numpy(dtype=bool)

        onset_results.append({
            "Horizon": f"{h}h",
            "Year": year,
            "Onsets": int(onsets.sum()),
            "Detected Onsets": int(pred[onsets].sum()),
            "Onset Recall": round(pred[onsets].mean(), 3) if onsets.sum() else 0})

display(pd.DataFrame(onset_results))

,Horizon,Year,Onsets,Detected Onsets,Onset Recall
0,48h,2022,11,3,0.273
1,48h,2023,7,2,0.286
2,48h,2024,5,1,0.200
3,72h,2022,11,1,0.091
4,72h,2023,7,1,0.143
5,72h,2024,5,3,0.600


## 5. Choosing a threshold without using the test year
### Tune the 48h Logistic Regression threshold on 2023

Fit the 48h model using 2021–2022 data, then try a fixed range of decision thresholds on **2023 validation data**. Select the threshold with the highest validation F1-score (0.20 here) rather than choosing it after inspecting 2024 test outcomes.

In [20]:
# Threshold tuning - Logistic Regression 48h

features = chunks_new.columns.drop(["target_date", "flood_flag"])

train = chunks_extended[chunks_extended["label_date"].dt.year <= 2022].copy()
val = chunks_extended[chunks_extended["label_date"].dt.year == 2023].copy()

# Avoid overlap between training labels and validation period
train = train[train["label_date"] + pd.Timedelta(hours=48) < val["label_date"].min()]

scaler = StandardScaler()
X_train = scaler.fit_transform(train[features])
X_val = scaler.transform(val[features])

model = LogisticRegression(max_iter=1000)
model.fit(X_train, train["flood_48h"])

prob = model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.1, 0.91, 0.05)

best_threshold = max(thresholds, key=lambda t: f1_score(val["flood_48h"], prob >= t, zero_division=0))

print("Best threshold:", round(best_threshold, 2))
print("F1 with threshold 0.5:", round(f1_score(val["flood_48h"], prob >= 0.5), 3))
print("F1 with selected threshold:", round(f1_score(val["flood_48h"], prob >= best_threshold), 3))

Best threshold: 0.2
F1 with threshold 0.5: 0.4
F1 with selected threshold: 0.417


### Evaluate standard vs selected threshold on 2024

Refit Logistic Regression using data through 2023 and evaluate SW90 during 2024 with both the default **0.50** threshold and the validation-selected **0.20** threshold. Lowering the threshold increases detected positive labels but also raises the number of false-positive daily flags.

In [21]:
# Final evaluation - 48h Logistic Regression

train = chunks_extended[chunks_extended["label_date"].dt.year <= 2023].copy()
test = chunks_extended[chunks_extended["label_date"].dt.year == 2024].copy()

# Avoid overlap with the test period
train = train[train["label_date"] + pd.Timedelta(hours=48) < test["label_date"].min()]

features = chunks_new.columns.drop(["target_date", "flood_flag"])

scaler_final = StandardScaler()
X_train = scaler_final.fit_transform(train[features])
X_test = scaler_final.transform(test[features])

model_final = LogisticRegression(max_iter=1000)
model_final.fit(X_train, train["flood_48h"])

prob = model_final.predict_proba(X_test)[:, 1]
y_test = test["flood_48h"]

# Compare both thresholds
for threshold in [0.5, 0.2]:
    pred = (prob >= threshold).astype(int)

    print("\nThreshold:", threshold)
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, zero_division=0))


Threshold: 0.5
Confusion Matrix:
 [[344   9]
 [  8   4]]
              precision    recall  f1-score   support

         0.0       0.98      0.97      0.98       353
         1.0       0.31      0.33      0.32        12

    accuracy                           0.95       365
   macro avg       0.64      0.65      0.65       365
weighted avg       0.96      0.95      0.95       365


Threshold: 0.2
Confusion Matrix:
 [[330  23]
 [  3   9]]
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96       353
         1.0       0.28      0.75      0.41        12

    accuracy                           0.93       365
   macro avg       0.64      0.84      0.69       365
weighted avg       0.97      0.93      0.94       365



### Check whether threshold tuning captures more onsets

For the same 2024 observations, compare the number of newly positive GFMS labels detected at both thresholds. The change from **1/5 to 4/5 onsets** is informative but based on only five transitions.

In [22]:
# Flood onsets - 2024

data = sw90_horizons.sort_values("date").copy()
data["onset"] = (data["flood_within_48h"].shift(1) == 0) & (data["flood_within_48h"] == 1)

onsets = test["label_date"].map(data.set_index("date")["onset"]).fillna(False).to_numpy()

for threshold in [0.5, 0.2]:
    pred = (prob >= threshold).astype(int)
    print(f"Threshold {threshold}: {pred[onsets].sum()} / {onsets.sum()} onsets detected")

Threshold 0.5: 1 / 5 onsets detected
Threshold 0.2: 4 / 5 onsets detected


## 6. Uncertainty and risk categories
### Month-block bootstrap for Recall uncertainty

Resample **whole months**, not individual daily rows, to partially preserve within-month dependence between neighboring days. Calculate empirical 95% percentile intervals for Recall at the two thresholds. The broad intervals emphasize that there are very few positive labels in the 2024 test period; they are not proof that one threshold will reliably outperform another in future years.

In [23]:
# Month-block bootstrap - 48h, 2024

months = test["label_date"].dt.month.to_numpy()
y = y_test.to_numpy()
rng = np.random.default_rng(42)

for threshold in [0.5, 0.2]:
    recalls = []

    for i in range(1000):
        sample_months = rng.choice(np.unique(months), size=12, replace=True)
        idx = np.concatenate([np.where(months == m)[0] for m in sample_months])

        if y[idx].sum() == 0:
            continue

        pred = (prob[idx] >= threshold).astype(int)
        recalls.append(recall_score(y[idx], pred))

    ci = np.percentile(recalls, [2.5, 97.5])
    print(f"Threshold {threshold}: 95% CI for Recall = {ci.round(3)}")

Threshold 0.5: 95% CI for Recall = [0.    0.771]
Threshold 0.2: 95% CI for Recall = [0.081 1.   ]


### 48h probability bands (risk gauge)

Group predicted 48h scores into four model-defined categories: **Low (<5%)**, **Elevated (5–<20%)**, **High (20–<50%)**, and **Very high (≥50%)**. Count test days, positive GFMS labels, and onsets within each band. These are bands of *model output*, not independently calibrated probabilities of real-world flooding.

In [24]:
# Flood Risk Gauge - 48h, 2024

gauge = test[["label_date", "flood_48h"]].copy()
gauge["Probability"] = prob

# Identify onsets
flags = sw90_horizons.set_index("date")["flood_within_48h"]
onset_flags = (flags.shift(1) == 0) & (flags == 1)
gauge["Onset"] = gauge["label_date"].map(onset_flags).fillna(False)

# Create risk bands
gauge["Risk Band"] = pd.cut(gauge["Probability"], bins=[0, 0.05, 0.20, 0.50, 1.01], labels=["Low", "Elevated", "High", "Very high"], right=False)

# Results by risk band
gauge_summary = gauge.groupby("Risk Band", observed=False).agg(Days=("flood_48h", "size"),Floods=("flood_48h", "sum"),Onsets=("Onset", "sum"))

gauge_summary["Flood Rate (%)"] = (gauge_summary["Floods"] / gauge_summary["Days"] * 100).round(2)

display(gauge_summary)

,Days,Floods,Onsets,Flood Rate (%)
Risk Band,,,,
Low,251,2.0,1,0.80
Elevated,82,1.0,0,1.22
High,19,5.0,3,26.32
Very high,13,4.0,1,30.77


### Compare the risk gauge across 24h, 48h, and 72h

Fit a model for each additional horizon and group SW90's 2024 scores using the same four cutoffs. Report the share of 0→1 GFMS transitions appearing in the High/Very high bands, alongside how many days those bands cover. This checks concentration of labels, not verified operational warning accuracy.

In [25]:
# Risk Gauge - 24h and 72h GFMS flags

features = chunks_new.columns.drop(["target_date", "flood_flag"])

for h in [24, 72]:
    col = "flood_flag" if h == 24 else "flood_72h"

    # Train and test
    train = chunks_extended[chunks_extended["label_date"].dt.year <= 2023].copy()
    test = chunks_extended[chunks_extended["label_date"].dt.year == 2024].copy()
    train = train[train["label_date"] + pd.Timedelta(hours=h) < test["label_date"].min()]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[features])
    X_test = scaler.transform(test[features])

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, train[col])

    # Predicted probabilities
    gauge = test[["label_date", col]].copy()
    gauge["Probability"] = model.predict_proba(X_test)[:, 1]

    # Identify positive-flag onsets
    flags = sw90_horizons.sort_values("date").set_index("date")[f"flood_within_{h}h"]
    onsets = (flags.shift(1) == 0) & (flags == 1)
    gauge["Onset"] = gauge["label_date"].map(onsets).fillna(False)

    # Risk bands
    gauge["Risk Band"] = pd.cut(gauge["Probability"], bins=[0, 0.05, 0.20, 0.50, 1.01],
                                labels=["Low", "Elevated", "High", "Very high"], right=False)

    # Summary
    summary = gauge.groupby("Risk Band", observed=False).agg(Days=(col, "size"),Floods=(col, "sum"),Onsets=("Onset", "sum"))

    summary["Flood Rate (%)"] = (summary["Floods"] / summary["Days"] * 100).round(2)

    print(f"\nRisk Gauge - {h}h (2024)")
    display(summary)

    total = int(summary["Onsets"].sum())
    high = int(summary.loc[["High", "Very high"], "Onsets"].sum())
    low = int(summary.loc["Low", "Onsets"])

    print(f"High + Very high onsets: {high}/{total} ({high / total * 100:.1f}%)")
    print(f"Low onsets: {low}/{total} ({low / total * 100:.1f}%)")


Risk Gauge - 24h (2024)


,Days,Floods,Onsets,Flood Rate (%)
Risk Band,,,,
Low,288,1.0,1,0.35
Elevated,56,2.0,1,3.57
High,17,3.0,3,17.65
Very high,4,1.0,0,25.00


High + Very high onsets: 3/5 (60.0%)
Low onsets: 1/5 (20.0%)

Risk Gauge - 72h (2024)


,Days,Floods,Onsets,Flood Rate (%)
Risk Band,,,,
Low,194,3.0,1,1.55
Elevated,119,1.0,0,0.84
High,38,5.0,1,13.16
Very high,14,8.0,3,57.14


High + Very high onsets: 4/5 (80.0%)
Low onsets: 1/5 (20.0%)


## 7. Expanding from two stations to all 30 stations
### Load the full cleaned Bangladesh dataset

Import `Data_Bangaldesh_Clean.csv` and retain **2021–2024** for a separate multi-station experiment. The full set contains 30 stations, enabling a test of whether learning from other river locations improves predictions specifically for SW90.

In [26]:
# Load dataset with all stations
url_all = "https://raw.githubusercontent.com/Burzo572/dsb_project/main/data/Data_Bangaldesh_Clean.csv"

df_all = pd.read_csv(url_all, parse_dates=["date"])
df_all = df_all[df_all["date"].dt.year.between(2021, 2024)].copy()

print("Shape:", df_all.shape)
print("Number of stations:", df_all["station_id"].nunique())

print("\nRows per year:")
print(df_all.groupby(df_all["date"].dt.year).size())

print("\nMissing 48h labels:")
print(df_all["flood_within_48h"].isna().sum())

print("\nPositive 48h labels:")
print(df_all.groupby(df_all["date"].dt.year)["flood_within_48h"].sum())

Shape: (43830, 45)
Number of stations: 30

Rows per year:
date
2021    10950
2022    10950
2023    10950
2024    10980
dtype: int64

Missing 48h labels:
0

Positive 48h labels:
date
2021     756.0
2022    1008.0
2023     855.0
2024    1169.0
Name: flood_within_48h, dtype: float64


### Validate targets, duplicates, and core predictors

Confirm that SW90's 48h positive-label counts match the earlier refined dataset, that the four selected weather/hydrological predictors have no missing values, and that no station/date pair is duplicated. This makes the model comparison more traceable.

In [27]:
# Check SW90 in the full dataset
sw90_all = df_all[df_all["station_id"] == "SW90"].copy()

print("SW90 positive 48h labels by year:")
print(sw90_all.groupby(sw90_all["date"].dt.year)["flood_within_48h"].sum())

# Check missing values in important predictors
features_all = [
    "rainfall_local_mm",
    "rainfall_upstream_mm",
    "soil_moisture_local",
    "river_discharge_m3s"]

print("\nMissing values:")
print(df_all[features_all].isna().sum())

print("\nDuplicate station-date rows:")
print(df_all.duplicated(subset=["station_id", "date"]).sum())

SW90 positive 48h labels by year:
date
2021    23.0
2022    31.0
2023    19.0
2024    12.0
Name: flood_within_48h, dtype: float64

Missing values:
rainfall_local_mm       0
rainfall_upstream_mm    0
soil_moisture_local     0
river_discharge_m3s     0
dtype: int64

Duplicate station-date rows:
0


### Generate three-day windows separately for each station

Use grouped shifts to build **12 predictors per row** (four measurements × three days). Grouping by `station_id` prevents one station's measurements from being used as the previous day's observations for another station. The first two dates per station are removed because they lack a complete three-day history.

In [28]:
# Prepare 3-day sliding windows for all 30 stations

all_station_model = df_all.sort_values(["station_id", "date"]).copy()

base_features = [
    "rainfall_local_mm",
    "rainfall_upstream_mm",
    "soil_moisture_local",
    "river_discharge_m3s"]

feature_cols_all = []

for feature in base_features:
    for lag in [2, 1, 0]:
        col = f"{feature}_day{3-lag}"
        all_station_model[col] = all_station_model.groupby("station_id")[feature].shift(lag)
        feature_cols_all.append(col)

all_station_model = all_station_model.dropna(subset=feature_cols_all + ["flood_within_48h"])

print("Shape:", all_station_model.shape)
print("Stations:", all_station_model["station_id"].nunique())
print("Number of predictors:", len(feature_cols_all))

Shape: (43770, 57)
Stations: 30
Number of predictors: 12


### Chronological training and SW90-only testing

Reserve 2024 for SW90 evaluation while training on earlier data from the available stations. The two-day gap before 2024 is a conservative boundary for the nominal 48h label. Both the single-station and 30-station models will be tested on **the same 365 SW90 rows** (12 positive labels).

In [29]:
# Train: all stations, 2021-2023
# Test: SW90 only, 2024

train_all = all_station_model[all_station_model["date"] + pd.Timedelta(days=2) < pd.Timestamp("2024-01-01")].copy()

test_sw90_all = all_station_model[
    (all_station_model["station_id"] == "SW90") &
    (all_station_model["date"] >= "2024-01-01") &
    (all_station_model["date"] < "2024-12-31")].copy()

print("Train rows:", len(train_all))
print("Train stations:", train_all["station_id"].nunique())

print("Test rows:", len(test_sw90_all))
print("Test positives:", int(test_sw90_all["flood_within_48h"].sum()))

Train rows: 32730
Train stations: 30
Test rows: 365
Test positives: 12


### Logistic Regression: SW90-only vs all 30 stations

Compare two Logistic Regression models with **identical 12 predictors and test data**, changing only the training source. This isolates the practical effect of adding stations more clearly than a comparison with the earlier 18-feature SW90+SW17 model. On this test, adding stations did not improve F1-score or Average Precision.

In [30]:
# Logistic Regression - SW90 vs All 30 Stations

results_all = []

X_test = test_sw90_all[feature_cols_all]
y_test = test_sw90_all["flood_within_48h"]

for name, train in [
    ("SW90 only", train_all[train_all["station_id"] == "SW90"]),
    ("All 30 stations", train_all)]:
    X_train = train[feature_cols_all]
    y_train = train["flood_within_48h"]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)

    pred = model.predict(X_test_scaled)
    prob = model.predict_proba(X_test_scaled)[:, 1]

    results_all.append({
        "Model": name,
        "Train Rows": len(train),
        "Detected": int(((pred == 1) & (y_test == 1)).sum()),
        "False Alarms": int(((pred == 1) & (y_test == 0)).sum()),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "PR-AUC (AP)": average_precision_score(y_test, prob)})

display(pd.DataFrame(results_all).round(3))

,Model,Train Rows,Detected,False Alarms,Precision,Recall,F1,PR-AUC (AP)
0,SW90 only,1091,2,1,0.667,0.167,0.267,0.397
1,All 30 stations,32730,2,3,0.400,0.167,0.235,0.326


### Check onset detection for both Logistic Regression models

Count how many of SW90's five 48h label onsets each model detects in 2024. Both models have zero onset detections at the default threshold, showing why ordinary classification metrics alone are insufficient.

In [31]:
# Compare onset detection - SW90 vs All 30 Stations

sw90_flags = all_station_model[all_station_model["station_id"] == "SW90"].sort_values("date").copy()

sw90_flags["onset"] = ((sw90_flags["flood_within_48h"].shift(1) == 0) &(sw90_flags["flood_within_48h"] == 1))

onsets = test_sw90_all["date"].map(sw90_flags.set_index("date")["onset"]).fillna(False).to_numpy(dtype=bool)

for name, train in [
    ("SW90 only", train_all[train_all["station_id"] == "SW90"]),
    ("All 30 stations", train_all)]:
    scaler = StandardScaler()

    X_train = scaler.fit_transform(train[feature_cols_all])
    X_test = scaler.transform(test_sw90_all[feature_cols_all])

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, train["flood_within_48h"])

    pred = model.predict(X_test)

    print(f"{name}: {pred[onsets].sum()} / {onsets.sum()} onsets detected")

SW90 only: 0.0 / 5 onsets detected
All 30 stations: 0.0 / 5 onsets detected


## 8. Station-relative features and HistGradientBoosting
### Normalize discharge relative to each station

Add features that compare each day's log discharge with that station's **median log discharge calculated only on training data**. Also include a day-to-day discharge change. This creates **16 total predictors** and addresses the fact that the same absolute discharge can mean different things at different stations.

In [32]:
# Station-relative river discharge

train_hgb = train_all.copy()
test_hgb = test_sw90_all.copy()

# Typical discharge calculated ONLY from training data
station_medians = train_hgb.groupby("station_id")["river_discharge_m3s_day3"].median()

for data in [train_hgb, test_hgb]:
    for day in [1, 2, 3]:
        col = f"river_discharge_m3s_day{day}"

        data[f"discharge_relative_day{day}"] = (
            np.log1p(data[col]) -
            np.log1p(data["station_id"].map(station_medians)))

# Change in discharge between consecutive days
for data in [train_hgb, test_hgb]:
    data["discharge_change"] = (
        data["river_discharge_m3s_day3"] -
        data["river_discharge_m3s_day2"])

# New feature list
features_hgb = feature_cols_all + [
    "discharge_relative_day1",
    "discharge_relative_day2",
    "discharge_relative_day3",
    "discharge_change"]

print("Number of predictors:", len(features_hgb))
print("Train rows:", len(train_hgb))
print("Test rows:", len(test_hgb))

print("\nMissing values:")
print(train_hgb[features_hgb].isna().sum().sum())
print(test_hgb[features_hgb].isna().sum().sum())

Number of predictors: 16
Train rows: 32730
Test rows: 365

Missing values:
0
0


### Train HistGradientBoosting on all 30 stations

Fit `HistGradientBoostingClassifier` on the pooled 2021–2023 training set and evaluate the 2024 SW90 target. A tree-based boosting model can represent nonlinear patterns and interactions that a simple linear classifier may miss. Report the confusion matrix, class-level performance, and Average Precision.

In [33]:
X_train = train_hgb[features_hgb]
y_train = train_hgb["flood_within_48h"]

X_test = test_hgb[features_hgb]
y_test = test_hgb["flood_within_48h"]

hgb_model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    random_state=42)

hgb_model.fit(X_train, y_train)

y_pred_hgb = hgb_model.predict(X_test)
y_prob_hgb = hgb_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_hgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hgb, zero_division=0))

print("PR-AUC (AP):", round(average_precision_score(y_test, y_prob_hgb), 3))

Confusion Matrix:
[[352   1]
 [ 10   2]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      1.00      0.98       353
         1.0       0.67      0.17      0.27        12

    accuracy                           0.97       365
   macro avg       0.82      0.58      0.63       365
weighted avg       0.96      0.97      0.96       365

PR-AUC (AP): 0.404


### Evaluate HGB onset detection

Separately count new positive-flag transitions found by HGB at its default 0.50 threshold. Even when Precision is relatively high, zero detected onsets means the classifier is still missing the beginning of positive-label periods.

In [34]:
# Flood Onset Detection - HGB

onset_dates = sw90_flags.set_index("date")["onset"]
onsets_hgb = test_hgb["date"].map(onset_dates).fillna(False).to_numpy(dtype=bool)

print("Total Onsets:", onsets_hgb.sum())
print("Detected Onsets:", y_pred_hgb[onsets_hgb].sum())

Total Onsets: 5
Detected Onsets: 0.0


### Select an HGB decision threshold using 2023

Fit a separate HGB model on 2021–2022 and choose the threshold with the highest F1-score on SW90's **2023 validation set**. Station medians are recalculated only from the training years to avoid using validation data when defining relative discharge. The selected threshold in this experiment is **0.25**.

In [35]:
# HGB Threshold Tuning - Validation 2023

tune_train = all_station_model[all_station_model["date"] < "2022-12-30"].copy()

tune_val = all_station_model[
    (all_station_model["station_id"] == "SW90") &
    (all_station_model["date"] >= "2023-01-01") &
    (all_station_model["date"] < "2023-12-30")].copy()

# Calculate medians using training data only
medians = tune_train.groupby("station_id")["river_discharge_m3s_day3"].median()

for data in [tune_train, tune_val]:
    for day in [1, 2, 3]:
        col = f"river_discharge_m3s_day{day}"
        data[f"discharge_relative_day{day}"] = (
            np.log1p(data[col]) - np.log1p(data["station_id"].map(medians)))

    data["discharge_change"] = (data["river_discharge_m3s_day3"] -data["river_discharge_m3s_day2"])

# Train HGB
hgb_val = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    random_state=42)

hgb_val.fit(tune_train[features_hgb], tune_train["flood_within_48h"])

# Predict probabilities for 2023
val_prob = hgb_val.predict_proba(tune_val[features_hgb])[:, 1]
val_y = tune_val["flood_within_48h"]

# Find the best threshold
thresholds = np.arange(0.05, 0.51, 0.05)

results_threshold = []

for t in thresholds:
    pred = (val_prob >= t).astype(int)

    results_threshold.append({
        "Threshold": round(t, 2),
        "Precision": precision_score(val_y, pred, zero_division=0),
        "Recall": recall_score(val_y, pred, zero_division=0),
        "F1": f1_score(val_y, pred, zero_division=0)})

threshold_results = pd.DataFrame(results_threshold)

display(threshold_results.round(3))

,Threshold,Precision,Recall,F1
0,0.05,0.145,0.895,0.250
1,0.10,0.218,0.895,0.351
2,0.15,0.280,0.737,0.406
3,0.20,0.324,0.632,0.429
4,0.25,0.375,0.632,0.471
5,0.30,0.320,0.421,0.364
6,0.35,0.263,0.263,0.263
7,0.40,0.000,0.000,0.000
8,0.45,0.000,0.000,0.000
9,0.50,0.000,0.000,0.000


### Apply the chosen HGB threshold to 2024

Compare HGB predictions at the default threshold of 0.50 and the validation-selected threshold of 0.25, using the already-fitted 2021–2023 model. The lower threshold captures more positive flags and one onset, but produces more false alarms. This is a trade-off, not a universally superior decision rule.

In [36]:
# HGB - Final Threshold Evaluation (2024)

for threshold in [0.5, 0.25]:

    pred = (y_prob_hgb >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, pred))

    print("Precision:", round(precision_score(y_test, pred, zero_division=0), 3))
    print("Recall:", round(recall_score(y_test, pred, zero_division=0), 3))
    print("F1:", round(f1_score(y_test, pred, zero_division=0), 3))

    print("False Alarms:", int(((pred == 1) & (y_test == 0)).sum()))
    print("Detected Onsets:", int(pred[onsets_hgb].sum()), "/", int(onsets_hgb.sum()))


Threshold: 0.5
Confusion Matrix:
[[352   1]
 [ 10   2]]
Precision: 0.667
Recall: 0.167
F1: 0.267
False Alarms: 1
Detected Onsets: 0 / 5

Threshold: 0.25
Confusion Matrix:
[[343  10]
 [  8   4]]
Precision: 0.286
Recall: 0.333
F1: 0.308
False Alarms: 10
Detected Onsets: 1 / 5


## 9. Final dataset checks
### Missing-data indicators and label regime

Check the dataset's recorded missing-measurement indicators and confirm that the retained 48h labels are marked `observed`. These fields support data completeness checks, but **`observed` is a label-regime value, not proof of a gauge-confirmed physical flood**. The upstream source's preprocessing and exact timestamp logic still require verification.

In [37]:
# Check original missing-value indicators

missing_flags = [
    "rainfall_local_mm_missing",
    "rainfall_upstream_mm_missing",
    "soil_moisture_local_missing",
    "river_discharge_m3s_missing"]

display(all_station_model.groupby(all_station_model["date"].dt.year)[missing_flags].sum())

# Check 48h label regimes
print("\n48h label regimes:")
print(all_station_model["flood_within_48h_label_regime"].value_counts(dropna=False))

,rainfall_local_mm_missing,rainfall_upstream_mm_missing,soil_moisture_local_missing,river_discharge_m3s_missing
date,,,,
2021,0,0,0,0
2022,0,0,0,0
2023,0,0,0,0
2024,0,0,0,0



48h label regimes:
flood_within_48h_label_regime
observed    43770
Name: count, dtype: int64


---

## 10. Final conclusion — Modeling

### Project objective and approach

This modeling stage builds directly on the EDA: the stations **SW90 (Bahadurabad)** and **SW17 (Chilmari)** were selected for the initial comparison, and the key predictors were local rainfall, soil moisture, and river discharge over three consecutive days. A later experiment added upstream rainfall and pooled data from all 30 stations. The target throughout was a **GFMS-derived positive flood-related flag**, evaluated using the dataset's 24h, 48h, and 72h labels.

I used chronological splits instead of random train/test shuffling because the data are time series. I also compared the machine-learning models against **persistence**, **seasonal**, and **probabilistic** baselines. Since positive flags are rare, the analysis focused on **Precision, Recall, F1-score, Average Precision**, false alarms, and **onset detection** (0→1 label transitions), not accuracy alone.

### Main results

- **24h Logistic Regression (SW90 + SW17):** Walk-forward Average Precision was **0.308 (2022), 0.301 (2023), and 0.354 (2024)**, exceeding the month/previous-flag probabilistic baseline in each year (**0.252, 0.253, and 0.168**). However, onset detection remained limited: **4 of 25** onsets across 2022–2024.
- **Longer GFMS horizons:** The 48h and 72h labels contained more positive examples. The models captured some new positive-label transitions, but performance varied by year; a larger number of positive flags did not automatically mean better onset detection.
- **48h Logistic Regression with threshold tuning:** The decision threshold was selected as **0.20 using 2023 validation data**. On SW90 in 2024, this model detected **9/12 positive labels (Recall 75%)** and **4/5 onsets**, with **F1-score 0.409** and **23 false-positive daily flags**. At the default threshold of 0.50, it detected **4/12 positive labels** and **1/5 onsets**, with **F1-score 0.320** and **9 false positives**. Lowering the threshold therefore improved detection but increased false alarms.
- **Uncertainty and risk bands:** Month-block bootstrap Recall intervals were wide because the 2024 test contained only **12 positive 48h labels**. In the 48h risk-gauge analysis, **32 of 365 days** fell in the High/Very high model-score bands; those days contained **9/12 positive flags** and **4/5 onsets**. These score bands are **not calibrated flood probabilities**.
- **All-station modeling:** A 12-feature Logistic Regression trained on all 30 stations did **not** outperform the same model trained only on SW90 in the 2024 test (**F1 0.235 vs 0.267; AP 0.326 vs 0.397**). The **HistGradientBoostingClassifier** with station-relative discharge features achieved **AP 0.404** at the default threshold, but detected **0/5 onsets**. With a threshold of **0.25 selected on 2023**, HGB reached **Recall 0.333, F1 0.308**, detected **1/5 onsets**, and had **10 false-positive daily flags** in 2024.

### Interpretation and limitations

The main lesson is that **a more complex model or a larger training dataset is not automatically better**. The simpler 48h Logistic Regression with a validation-selected threshold captured more positive labels and new transitions than HGB in the 2024 comparison, but generated more false alarms. The models differ in feature sets and training design, so this is an **experimental comparison**, not proof that one algorithm is universally superior.

Results must be interpreted cautiously because the **positive class is small**, consecutive GFMS labels are dependent, and testing focused primarily on SW90. The 2024 period was also examined repeatedly during model development, which limits its status as an entirely untouched final test. Most importantly, the original source's **GFMS label-generation and timestamp conventions** have not been independently verified. A `target_date` stored one day after the final input observation does not itself prove true 24h/48h/72h advance warning, and `observed` in a label-regime column does not certify gauge-confirmed flooding. The all-station cleaned CSV should also be checked against source preprocessing for any information used across time boundaries.

### Final takeaway

This project demonstrates a complete time-aware Data Science workflow: **EDA → feature preparation → baselines → models → walk-forward evaluation → onset analysis → threshold selection → uncertainty and risk-score analysis → multi-station comparison**. It shows useful associations between hydrometeorological measurements and GFMS positive flags, while identifying clear limitations. Future work should verify source timestamps and target meaning, evaluate on genuinely new years or locations, test probability calibration, and confirm performance using independently observed flood events before considering operational early-warning use.

In [38]:
# Ridge (L2) vs Lasso (L1) - 48h Logistic Regression

features = chunks_new.columns.drop(["target_date", "flood_flag"])

train = chunks_extended[chunks_extended["label_date"].dt.year <= 2022].copy()
val = chunks_extended[chunks_extended["label_date"].dt.year == 2023].copy()

# Avoid overlap between training labels and validation
train = train[train["label_date"] + pd.Timedelta(hours=48) < val["label_date"].min()]

scaler = StandardScaler()

X_train = scaler.fit_transform(train[features])
X_val = scaler.transform(val[features])

y_train = train["flood_48h"]
y_val = val["flood_48h"]

results_regularization = []

for name, penalty in [("Ridge (L2)", "l2"), ("Lasso (L1)", "l1")]:

    for C in [0.1, 1, 10]:

        model = LogisticRegression(
            penalty=penalty,
            C=C,
            solver="liblinear",
            max_iter=1000)

        model.fit(X_train, y_train)
        prob = model.predict_proba(X_val)[:, 1]

        for threshold in [0.15, 0.20, 0.25, 0.30, 0.35]:

            pred = (prob >= threshold).astype(int)

            results_regularization.append({
                "Model": name,
                "C": C,
                "Threshold": threshold,
                "Detected": int(((pred == 1) & (y_val == 1)).sum()),
                "False Alarms": int(((pred == 1) & (y_val == 0)).sum()),
                "Recall": recall_score(y_val, pred, zero_division=0),
                "F1": f1_score(y_val, pred, zero_division=0)})

results_reg = pd.DataFrame(results_regularization)

# Show candidates that preserve at least 60% Recall
display(
    results_reg[results_reg["Recall"] >= 0.60]
    .sort_values(["False Alarms", "F1"], ascending=[True, False])
    .head(10)
    .round(3))

,Model,C,Threshold,Detected,False Alarms,Recall,F1
16,Lasso (L1),0.1,0.20,13,19,0.684,0.510
1,Ridge (L2),0.1,0.20,12,27,0.632,0.414
20,Lasso (L1),1.0,0.15,12,28,0.632,0.407
15,Lasso (L1),0.1,0.15,15,34,0.789,0.441
0,Ridge (L2),0.1,0.15,17,48,0.895,0.405


In [40]:
# Final comparison: Logistic vs Ridge vs Lasso (48h)

features_reg = chunks_new.columns.drop(["target_date", "flood_flag"])

train_reg = chunks_extended[chunks_extended["label_date"].dt.year <= 2023].copy()

test_reg = chunks_extended[
    (chunks_extended["label_date"] >= "2024-01-01") &
    (chunks_extended["label_date"] < "2024-12-31")].copy()

# Keep a 48h gap before the test period
train_reg = train_reg[train_reg["label_date"] + pd.Timedelta(hours=48) < test_reg["label_date"].min()]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_reg[features_reg])
X_test = scaler.transform(test_reg[features_reg])

y_train = train_reg["flood_48h"]
y_test = test_reg["flood_48h"].to_numpy()

# Find new positive GFMS flags
flags = sw90_horizons.sort_values("date").set_index("date")["flood_within_48h"]
onset_flags = (flags.shift(1) == 0) & (flags == 1)
onsets = test_reg["label_date"].map(onset_flags).fillna(False).to_numpy(dtype=bool)

models = {
    "Original Logistic": LogisticRegression(max_iter=1000),
    "Ridge (L2)": LogisticRegression(penalty="l2", C=0.1, solver="liblinear", max_iter=1000),
    "Lasso (L1)": LogisticRegression(penalty="l1", C=0.1, solver="liblinear", max_iter=1000)}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.20).astype(int)

    results.append({
        "Model": name,
        "Detected": int(((pred == 1) & (y_test == 1)).sum()),
        "False Alarms": int(((pred == 1) & (y_test == 0)).sum()),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "PR-AUC (AP)": average_precision_score(y_test, prob),
        "Detected Onsets": f"{int(pred[onsets].sum())}/{int(onsets.sum())}"})

print("Test days:", len(test_reg))
print("Positive flags:", int(y_test.sum()))

display(pd.DataFrame(results).round(3))

Test days: 365
Positive flags: 12


,Model,Detected,False Alarms,Precision,Recall,F1,PR-AUC (AP),Detected Onsets
0,Original Logistic,9,23,0.281,0.750,0.409,0.314,4/5
1,Ridge (L2),9,24,0.273,0.750,0.400,0.268,4/5
2,Lasso (L1),7,18,0.280,0.583,0.378,0.321,2/5


### Additional Experiment — L1 and L2 Regularization

To investigate whether regularization could reduce false alarms, I compared the original Logistic Regression model with stronger L2 (Ridge-style) and L1 (Lasso-style) regularization for the 48-hour GFMS target.

Regularization settings were selected using 2023 validation data and evaluated on SW90 during 2024.

The L2 model detected 9 of 12 positive flags, matching the original model, but produced one additional false alarm.

The L1 model reduced false alarms from 23 to 18. However, it detected only 7 of 12 positive flags, compared with 9 for the original model. More importantly, onset detection decreased from 4 of 5 to 2 of 5.

**Conclusion:** L1 regularization reduced false alarms but also reduced sensitivity to positive flags and new onsets. For the objective of detecting more positive flood-related signals, the original Logistic Regression with a threshold of 0.20 remains the preferred exploratory configuration.

These findings are based on a small test set, and further independent validation is required.